# 005 · Timing and Profiling

> **Standard Library › Testing & Debugging** · Python 3.14

Before optimising anything you need numbers: *how long* does this take, *where* is the time going, and *what* is using the memory? The standard library answers each question with a dedicated module — `timeit` for micro-benchmarks, `cProfile` + `pstats` for CPU profiling, and `tracemalloc` for memory.

**In this notebook you will learn:**

- how to benchmark small snippets with `timeit` (the BenchmarkDotNet / `Stopwatch` of Python)
- the IPython `%timeit` and `%%timeit` magics for quick measurements in notebooks
- how to find hot spots with `cProfile` and read the report with `pstats`
- how to track memory allocations and find leaks with `tracemalloc`
- the usual pitfalls: noisy measurements, profiler overhead, and optimising the wrong thing

## .NET mental model

| Question | .NET | Python (stdlib) |
|---|---|---|
| "How long did this take?" | `Stopwatch.StartNew()` / `Stopwatch.GetTimestamp()` | `time.perf_counter()` |
| "Which version is faster?" (micro-benchmark) | BenchmarkDotNet (`[Benchmark]`) | `timeit`, `%timeit` |
| "Where is the CPU time going?" | Visual Studio Profiler, `dotnet-trace`, dotTrace | `cProfile` + `pstats` (`python -m cProfile`) |
| "What is allocating memory?" | VS memory usage tool, `dotnet-counters`, `dotnet-gcdump`, dotMemory | `tracemalloc` |
| Low-overhead sampling profiler | `dotnet-trace` (sampling), PerfView | third-party tools such as py-spy (not in the stdlib) |

`cProfile` is a **deterministic** (tracing) profiler: it records every function call and return. That makes call counts exact but adds overhead to every call — so use `timeit` for "how fast", and `cProfile` for "where".

## Stopwatch-style timing with `perf_counter`

For a one-off measurement, `time.perf_counter()` is the high-resolution monotonic clock — the equivalent of `Stopwatch` *(see Standard Library › Dates & Times › 004 · Clocks and Measuring Time)*:

```csharp
var sw = Stopwatch.StartNew();
var total = Enumerable.Range(0, 1_000_000).Sum(i => (long)i * i);
Console.WriteLine($"{sw.Elapsed.TotalMilliseconds:F1} ms");
```

In [ ]:
import time

start = time.perf_counter()
total = sum(i * i for i in range(1_000_000))
elapsed = time.perf_counter() - start
print(f"total={total}, took {elapsed * 1000:.1f} ms")   # timing varies by machine

A single measurement is noisy (caches, CPU frequency, other processes, the garbage collector). That is what `timeit` is for.

## Micro-benchmarks with `timeit`

`timeit` runs a statement **many times** in a loop and returns the **total** seconds. It minimises overhead and temporarily disables the garbage collector during the measurement. You can pass either a **callable** or a **string of code**:

```csharp
// BenchmarkDotNet
[Benchmark] public string Join()   => string.Join(",", items);
[Benchmark] public string Concat() { var s = ""; foreach (var i in items) s += i + ","; return s; }
```

In [ ]:
import timeit

items = [str(i) for i in range(1_000)]

def with_join():
    return ",".join(items)

def with_concat():
    s = ""
    for item in items:
        s += item + ","
    return s

n = 200
for func in (with_join, with_concat):
    seconds = timeit.timeit(func, number=n)            # callable: no setup string needed
    print(f"{func.__name__:12} {seconds / n * 1e6:8.1f} µs per call")

With a **string** statement, the code runs in a fresh namespace: anything it needs must come from the `setup` string or be passed in with `globals=`:

In [ ]:
setup = "data = list(range(10_000)); lookup = set(data)"

t_list = timeit.timeit("9_999 in data", setup=setup, number=500)
t_set = timeit.timeit("9_999 in lookup", setup=setup, number=500)
print(f"list membership: {t_list / 500 * 1e6:7.2f} µs")
print(f"set membership:  {t_set / 500 * 1e6:7.2f} µs")      # far faster: hash lookup

# Using names defined in the notebook instead of a setup string:
print(timeit.timeit("with_join()", globals=globals(), number=100) > 0)   # True

⚠️ **Gotcha:** a string statement that refers to a notebook variable without `globals=` (or `setup`) fails with `NameError` — the statement does *not* see your variables:

In [ ]:
try:
    timeit.timeit("with_join()", number=1)
except NameError as e:
    print(f"{type(e).__name__}: {e}")

### `repeat`: take the minimum

`timeit.repeat` runs the whole measurement several times and returns a list. Report the **minimum** (the run least disturbed by other activity); the spread tells you how noisy the machine is. `Timer.autorange()` picks a sensible `number` for you, like BenchmarkDotNet's pilot stage.

In [ ]:
runs = timeit.repeat(with_join, number=200, repeat=5)
print(f"best of 5: {min(runs) / 200 * 1e6:.1f} µs per call")
print(f"spread:    {(max(runs) - min(runs)) / min(runs):.0%}")

loops, seconds = timeit.Timer(with_join).autorange()     # grows number until >= 0.2 s
print(f"autorange chose number={loops}")

From a terminal, `python -m timeit` does the loop-count selection and repetition for you:

```bash
python -m timeit -s "data = list(range(10_000))" "9_999 in data"
# 5000 loops, best of 5: 43.1 usec per loop        (example output)
```

## `%timeit` and `%%timeit` in notebooks

IPython adds magics that wrap `timeit` and print a readable summary (mean ± standard deviation). They see notebook variables automatically.

- `%timeit stmt` — time one line
- `%%timeit` as the **first line of a cell** — time the whole cell (lines after it); `%%timeit setup_line` puts setup on the magic line
- `-n N` loops per run, `-r R` runs; `-o` returns a result object you can inspect

They are IPython features, not Python syntax: they will not work in a `.py` file.

In [ ]:
%timeit -n 200 -r 3 with_join()

In [ ]:
%%timeit -n 50 -r 3 data = list(range(5_000))
sorted(data, reverse=True)

In [ ]:
result = %timeit -o -q -n 100 -r 3 with_concat()
print(f"best: {result.best * 1e6:.1f} µs, runs: {len(result.all_runs)}")

⚠️ **Gotcha:** micro-benchmarks lie easily. Measure realistic input sizes, compare implementations under the same conditions, and remember that in real programs I/O, algorithms and data structures usually matter far more than shaving nanoseconds off a loop. Profile first, then optimise what the profile shows.

## CPU profiling with `cProfile` and `pstats`

`cProfile` records every call: how often each function ran, its **own** time (`tottime`) and the time **including** everything it called (`cumtime`). Here is a deliberately slow program — it checks for duplicates with list membership (O(n²)) and recomputes a value in a hot loop:

In [ ]:
import cProfile
import pstats

def normalise(sku: str) -> str:
    return sku.strip().upper()

def find_duplicates(skus: list[str]) -> list[str]:
    seen, dupes = [], []
    for sku in skus:
        key = normalise(sku)
        if key in seen:                 # O(n) list scan inside a loop -> O(n²)
            dupes.append(key)
        else:
            seen.append(key)
    return dupes

skus = [f" sku-{i % 3_000} " for i in range(6_000)]
print(len(find_duplicates(skus)))       # 3000

`cProfile.Profile` is a context manager. Feed the result to `pstats.Stats`, then `strip_dirs()` (shorter file names), `sort_stats()` and `print_stats(...)` — a number limits the rows, a string is a regex filter on the `filename:lineno(function)` column.

In a notebook the profile can also pick up calls made by the kernel's own machinery while the profiler is active (for example `socket.py` sends that deliver cell output), so we filter the report down to our code:

In [ ]:
with cProfile.Profile() as profiler:
    find_duplicates(skus)

stats = pstats.Stats(profiler).strip_dirs().sort_stats(pstats.SortKey.TIME)
stats.print_stats(r"find_duplicates|normalise|'strip'|of 'list'")
print("done")

How to read it — the columns map closely onto a Visual Studio CPU-usage report:

| Column | Meaning | VS profiler term |
|---|---|---|
| `ncalls` | number of calls (`a/b` = total/primitive for recursion) | Hit count |
| `tottime` | time spent in the function itself, excluding callees | Self CPU |
| `percall` | `tottime / ncalls` | — |
| `cumtime` | time including all callees | Total CPU |
| `filename:lineno(function)` | where it is defined | Function name |

Here `find_duplicates` itself dominates `tottime` (the `in seen` list scans run inside it), while `normalise` is called 6 000 times but is cheap. Fixing the data structure (a `set`) is the real win:

In [ ]:
def find_duplicates_fast(skus: list[str]) -> list[str]:
    seen: set[str] = set()
    dupes = []
    for sku in skus:
        key = normalise(sku)
        if key in seen:
            dupes.append(key)
        else:
            seen.add(key)
    return dupes

slow = min(timeit.repeat(lambda: find_duplicates(skus), number=1, repeat=3))
fast = min(timeit.repeat(lambda: find_duplicates_fast(skus), number=1, repeat=3))
print(find_duplicates_fast(skus) == find_duplicates(skus))   # True — same result
print(f"speed-up: about {slow / fast:.0f}x")                 # varies by machine

### Sorting, filtering and saving profiles

- `sort_stats` keys: `"cumulative"`, `"tottime"` / `"time"`, `"calls"`, `"name"` … (or the `pstats.SortKey` enum)
- `print_stats(10)` limits to 10 rows; `print_stats("normalise")` filters by a regex on the function/file name
- `print_callers("normalise")` / `print_callees(...)` show who calls what (like the VS "Caller/Callee" view)
- `cProfile.run("stmt", filename)` profiles a statement and saves the raw data to a file, like saving a `.nettrace`/`.diagsession`; load it later with `pstats.Stats(filename)`
- `get_stats_profile()` returns the data as objects (`FunctionProfile` with `ncalls`, `tottime`, `cumtime`, …) for use in code, e.g. a performance regression check

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    out = str(Path(tmp) / "dupes.prof")
    cProfile.run("find_duplicates(skus)", out)          # runs in __main__'s namespace
    print("saved profile file:", Path(out).name)
    saved = pstats.Stats(out).strip_dirs()

saved.sort_stats("cumulative").print_callers("normalise")      # who calls normalise?
row = saved.get_stats_profile().func_profiles["normalise"]
print(f"normalise: ncalls={row.ncalls}, cumtime={row.cumtime:.4f}s")

From the command line (the usual way to profile a whole program):

```bash
python -m cProfile -s cumulative app.py          # print a report sorted by cumulative time
python -m cProfile -o app.prof app.py            # save for later analysis
python -m pstats app.prof                        # interactive report browser (sort, stats, callers…)
```

In IPython/Jupyter, `%prun find_duplicates(skus)` profiles one statement and `%%prun` a whole cell.

⚠️ **Gotcha:** `cProfile`'s per-call overhead inflates the cost of code that makes many tiny function calls. Use the profile to find **where** time goes, then confirm improvements with `timeit` on the unprofiled code.

## Memory: `tracemalloc`

`tracemalloc` hooks Python's memory allocator and records **where** (file and line) each block was allocated. Typical workflow, much like taking two snapshots in the Visual Studio memory tool and diffing them:

1. `tracemalloc.start()`
2. take a snapshot, run the suspicious code, take another snapshot
3. `snapshot2.compare_to(snapshot1, "lineno")` → sorted list of the biggest differences
4. `tracemalloc.stop()`

In [ ]:
import inspect
import linecache
import tracemalloc

cache: dict[int, str] = {}

def fill_cache(n: int) -> None:
    for i in range(n):
        cache[i] = f"product-description-{i}" * 4      # the "leak": grows forever

this_cell = inspect.currentframe().f_code.co_filename   # to ignore allocations elsewhere

tracemalloc.start()
before = tracemalloc.take_snapshot()
fill_cache(20_000)
after = tracemalloc.take_snapshot()
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"traced now: {current / 1024:,.0f} KiB, peak: {peak / 1024:,.0f} KiB")

In [ ]:
only_this_cell = [tracemalloc.Filter(True, this_cell)]
diff = after.filter_traces(only_this_cell).compare_to(before.filter_traces(only_this_cell), "lineno")

for stat in diff[:2]:
    frame = stat.traceback[0]
    source = linecache.getline(frame.filename, frame.lineno).strip()
    print(f"+{stat.size_diff / 1024:7,.0f} KiB  {stat.count_diff:+6} blocks  line {frame.lineno}: {source}")

The top entry points straight at the line that keeps allocating. A real report would also show library code; the `Filter` above just hides allocations made outside this cell. Other useful calls:

| Call | Purpose |
|---|---|
| `snapshot.statistics("lineno")` | top allocations in one snapshot (no diff) |
| `snapshot.statistics("traceback")` | group by full call stack (start with `tracemalloc.start(25)` to record 25 frames) |
| `tracemalloc.get_traced_memory()` | `(current, peak)` bytes since `start()` |
| `tracemalloc.reset_peak()` | measure the peak of one phase |
| `python -X tracemalloc=25 app.py` | start tracing from interpreter startup |

**Tip:** `tracemalloc` measures Python allocations, not the process's total memory (RSS). For a quick "how big is this object?", `sys.getsizeof(obj)` gives the shallow size of one object only — it does not follow references.

In [ ]:
import sys

del cache                                              # release the "leaked" memory
print(sys.getsizeof([]), sys.getsizeof([0] * 1_000))   # shallow sizes in bytes (64-bit CPython)

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Measure elapsed time | `Stopwatch` | `time.perf_counter()` |
| Micro-benchmark | BenchmarkDotNet | `timeit.timeit(func, number=N)` / `timeit.repeat(...)` |
| Benchmark a snippet string | — | `timeit.timeit("stmt", setup="...", globals=globals())` |
| Benchmark from a shell | `dotnet run -c Release` + BenchmarkDotNet | `python -m timeit -s "setup" "stmt"` |
| Quick timing in a notebook | — | `%timeit -n 100 -r 3 expr`, `%%timeit` |
| CPU profile | VS Profiler, `dotnet-trace` | `cProfile.Profile()`, `python -m cProfile -o out.prof app.py` |
| Read a profile | VS report / PerfView | `pstats.Stats(p).strip_dirs().sort_stats("cumulative").print_stats(10)` |
| Memory allocations | VS memory tool, `dotnet-gcdump`, dotMemory | `tracemalloc` snapshots + `compare_to` |
| Object size | `Unsafe.SizeOf<T>()` (value types) | `sys.getsizeof(obj)` (shallow) |

## Exercises

1. Use `timeit.repeat` to compare `[x * 2 for x in range(10_000)]` with `list(map(lambda x: x * 2, range(10_000)))` (best of 5, `number=50`). Print the time per call in µs for each.
2. Write a recursive `fib(n)` (no caching) and profile `fib(20)` with `cProfile.Profile` + `pstats`. Print the stats sorted by `"calls"` and find how many times `fib` was called (hint: `stats.stats` is a dict; or read the `ncalls` column).
3. Use `tracemalloc` to measure the **peak** memory of building `[str(i) for i in range(100_000)]` versus summing `len(str(i))` with a generator expression over the same range. Use `tracemalloc.reset_peak()` between the two measurements.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import cProfile
import pstats
import timeit
import tracemalloc

# 1
for label, stmt in [("comprehension", lambda: [x * 2 for x in range(10_000)]),
                    ("map + lambda", lambda: list(map(lambda x: x * 2, range(10_000))))]:
    best = min(timeit.repeat(stmt, number=50, repeat=5))
    print(f"{label:14} {best / 50 * 1e6:8.1f} µs")

# 2
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

with cProfile.Profile() as profiler:
    fib(20)
stats = pstats.Stats(profiler).strip_dirs().sort_stats("calls")
stats.print_stats(3)
for (filename, line, name), (cc, nc, tt, ct, callers) in stats.stats.items():
    if name == "fib":
        print("fib was called", nc, "times")     # 21891

# 3
tracemalloc.start()
strings = [str(i) for i in range(100_000)]
_, peak_list = tracemalloc.get_traced_memory()
del strings
tracemalloc.reset_peak()
total = sum(len(str(i)) for i in range(100_000))
_, peak_gen = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"list peak: {peak_list / 1024:,.0f} KiB, generator peak: {peak_gen / 1024:,.0f} KiB")
```

</details>